# OMS Multimodal SDK 全量功能示例

本 Notebook 演示 SDK **从 rosbag 到中间产物再到最终产物** 的完整链路：

| 阶段 | 产物 | 说明 |
|------|------|------|
| 本地解析 | `audio.wav` | clip 级拼接音频 |
| 本地解析 | `acoustic_panel.png` | Mel / STFT 频谱图（VL-embedding image 输入） |
| 本地解析 | `mel_matrix.csv` (+ `.meta.json`) | Mel 矩阵数值导出 |
| 本地解析 | `clip_preview*.mp4` | 预览视频编码 |
| 云端 ASR | `asr_text` | 语音转写文本 |
| 云端打标 | `labels.jsonl` | Omni 场景摘要 + taxonomy 标签 |
| 云端向量 | `fusion_embeddings.jsonl` | 多模态融合向量 |

> **前提**：已在 `piplinesdk/` 执行 `pip install -e ".[mc]"`（或 `pip install -e .`），并配置好 DashScope 密钥。
>
> 推荐 Python **3.11 / 3.12**。

## 0. 环境与路径

下面会自动探测仓库根目录（兼容 Windows `D:\cursor_project\...` 与 Linux 工作区）。

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# 路径约定
#   本 notebook 位于：<repo>/pipeline/local_sdk_mc_test/
#   SDK 源码位于：    <repo>/piplinesdk/
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()
# 若从其它 cwd 打开，可强制指定：
# NOTEBOOK_DIR = Path(r"D:\cursor_project\rosbag_to_labels_pipline\pipeline\local_sdk_mc_test")

REPO_ROOT = NOTEBOOK_DIR.parent.parent if (NOTEBOOK_DIR.parent.parent / "piplinesdk").is_dir() else NOTEBOOK_DIR
SDK_ROOT = REPO_ROOT / "piplinesdk"
OUT_DIR = NOTEBOOK_DIR / "output"
WORK_DIR = OUT_DIR / "work"
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# 保证能 import 到 editable 安装；若未 pip install -e，则把源码加入 path
if str(SDK_ROOT) not in sys.path:
    sys.path.insert(0, str(SDK_ROOT))

# 优先加载 piplinesdk/.env，其次本目录 .env
from dotenv import load_dotenv

for env_path in (SDK_ROOT / ".env", NOTEBOOK_DIR / ".env"):
    if env_path.is_file():
        load_dotenv(env_path, override=False)
        print(f"loaded env: {env_path}")

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("REPO_ROOT    =", REPO_ROOT)
print("SDK_ROOT     =", SDK_ROOT)
print("OUT_DIR      =", OUT_DIR)
print("API_KEY set  =", bool(os.getenv("DASHSCOPE_API_KEY")))
print("WORKSPACE set=", bool(os.getenv("DASHSCOPE_WORKSPACE_ID")))

In [ ]:
# ---------------------------------------------------------------------------
# 【必改】指向你的 ROS1 .bag
# ---------------------------------------------------------------------------
BAG_PATH = Path(
    os.getenv(
        "SDK_DEMO_BAG",
        # 默认占位：请改成真实 bag，或设置环境变量 SDK_DEMO_BAG
        r"D:\cursor_project\rosbag_to_labels_pipline\rosbag\output.bag",
    )
).expanduser().resolve()

# 演示时只跑前 N 个 clip，避免一次打满 API；全量可改成 None
MAX_CLIPS = int(os.getenv("SDK_DEMO_MAX_CLIPS", "1"))

print("BAG_PATH  =", BAG_PATH)
print("exists    =", BAG_PATH.is_file())
print("MAX_CLIPS =", MAX_CLIPS)
assert BAG_PATH.is_file(), f"找不到 bag，请修改 BAG_PATH: {BAG_PATH}"

## 1. 创建 SDK Client

- `AcousticPanelConfig.export_mel_matrix=True`：extract 时写出 Mel 矩阵与 `mel_feature_text`
- `ClipVideoConfig.enabled=True`：生成预览 MP4
- `model_backend="api"`：本地/ECS 走 DashScope API（Omni / ASR / Embedding）

In [ ]:
from oms_multimodal import (
    __version__,
    AcousticPanelConfig,
    AsrConfig,
    ClipConfig,
    ClipVideoConfig,
    OmsMultimodalClient,
    OutputConfig,
    bundled_taxonomy_path,
    inspect_bag,
)

print("oms_multimodal version:", __version__)

# Mel 面板 + 矩阵导出（打标 / 向量化 text 侧会吃 mel_feature_text）
acoustic_cfg = AcousticPanelConfig(
    panel_type="mel",
    n_mels=128,
    target_width=768,
    target_height=256,
    export_mel_matrix=True,   # 写出 mel_matrix.csv
    mel_matrix_csv=True,
    mel_matrix_npy=False,     # 需要二进制矩阵时改 True
    mel_feature_max_frames=32,
    mel_feature_max_chars=6000,
)

# 预览视频：默认编码全部相机（也可用环境变量 CLIP_VIDEO_* 覆盖）
video_cfg = ClipVideoConfig.from_env()
video_cfg.enabled = True
video_cfg.encode_all_cameras = True

asr_cfg = AsrConfig.from_env()
asr_cfg.enabled = True

client = OmsMultimodalClient(
    taxonomy_path=bundled_taxonomy_path(),  # wheel/源码自带 taxonomy
    work_dir=WORK_DIR,
    acoustic_panel_config=acoustic_cfg,
    clip_video_config=video_cfg,
    asr_config=asr_cfg,
    model_backend="api",
    storage_backend="local",
    load_dotenv=True,
)

clip_cfg = ClipConfig(
    min_sec=15.0,
    max_sec=20.0,
    sample_fps=1.0,
    max_clips=MAX_CLIPS,
)

print("taxonomy:", bundled_taxonomy_path())
print("work_dir:", WORK_DIR)

## 2. 查看 bag 内 topic（可选）

In [ ]:
topics = inspect_bag(BAG_PATH)
for t in topics:
    # TopicInfo: name / msgtype / modality / message_count
    print(f"{t.modality:6}  {t.message_count:6}  {t.name}  ({t.msgtype})")

## 3. 本地提取中间产物（不调云端模型）

`iter_clips()` 会在 `work_dir/clips/{clip_id}/` 写出：

- `audio.wav` — 音频本身
- `acoustic_panel.png` — Mel 频谱图
- `mel_matrix.csv` / `mel_matrix.meta.json` — Mel 矩阵
- `clip_preview*.mp4` — 视频编码

本步**不调用** ASR / Omni / Embedding。

In [ ]:
clips = list(client.iter_clips(BAG_PATH, clip_config=clip_cfg))
print(f"extracted {len(clips)} clip(s)")

assert clips, "未切出任何 clip，请检查 bag 时长 / topic / ClipConfig"

# 以第一个 clip 做后续逐步演示
clip = clips[0]
print("clip_id       =", clip.clip_id)
print("duration_sec  =", round(clip.duration_sec, 3))
print("frames(omni)  =", len(clip.frames))
print("video_frames  =", len(clip.video_frames))
print("emb_frames    =", len(clip.embedding_frames))

# ---- 中间产物路径一览 ----
print("\n=== 中间产物 ===")
print("audio.wav            =", getattr(clip.audio, "audio_path", None) if clip.audio else None)
print("acoustic_panel.png   =", clip.acoustic_panel_path)
print("mel_matrix.csv       =", clip.mel_matrix_path)
print("mel_matrix.meta.json =", clip.mel_matrix_meta_path)
print("mel_matrix.shape     =", clip.mel_matrix_shape)
print("clip_video_path      =", clip.clip_video_path)
print("clip_video_paths     =", clip.clip_video_paths)

# 持久化一份 clip 元数据，方便对照
meta_path = OUT_DIR / f"{clip.clip_id}_clip_meta.json"
meta_path.write_text(json.dumps(clip.to_meta(), ensure_ascii=False, indent=2), encoding="utf-8")
print("\nmeta saved ->", meta_path)

### 3.1 展示：音频 / Mel 图 / Mel 矩阵预览

In [ ]:
from IPython.display import Audio, Image, Markdown, display

# 1) 音频本身
if clip.audio and clip.audio.audio_path and Path(clip.audio.audio_path).is_file():
    wav_path = Path(clip.audio.audio_path)
    print("WAV:", wav_path, f"({wav_path.stat().st_size} bytes)")
    display(Audio(filename=str(wav_path)))
else:
    print("本 clip 无音频，跳过 Audio 预览")

# 2) Mel 频谱图 PNG
if clip.acoustic_panel_path and Path(clip.acoustic_panel_path).is_file():
    print("Mel PNG:", clip.acoustic_panel_path)
    display(Image(filename=clip.acoustic_panel_path))
else:
    print("无 acoustic_panel.png")

# 3) Mel 矩阵：读 meta + 前几行 CSV + feature text 摘要
if clip.mel_matrix_meta_path and Path(clip.mel_matrix_meta_path).is_file():
    meta = json.loads(Path(clip.mel_matrix_meta_path).read_text(encoding="utf-8"))
    print("mel meta:", json.dumps(meta, ensure_ascii=False, indent=2)[:800])

if clip.mel_matrix_path and Path(clip.mel_matrix_path).is_file():
    csv_path = Path(clip.mel_matrix_path)
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    print(f"\nmel_matrix.csv lines={len(lines)}  preview:")
    for line in lines[:5]:
        print(line[:160] + ("..." if len(line) > 160 else ""))

if clip.mel_feature_text:
    display(Markdown("**mel_feature_text（将注入打标/向量 text 侧）**"))
    print(clip.mel_feature_text[:1200])
    # 另存一份便于人工查看
    (OUT_DIR / f"{clip.clip_id}_mel_feature.txt").write_text(clip.mel_feature_text, encoding="utf-8")

### 3.2 展示：视频编码产物

In [ ]:
from IPython.display import Video

video_candidates: list[Path] = []
if clip.clip_video_paths:
    video_candidates.extend(Path(p) for p in clip.clip_video_paths.values())
if clip.clip_video_path:
    video_candidates.append(Path(clip.clip_video_path))

# 去重且仅保留存在的文件
seen = set()
videos = []
for p in video_candidates:
    key = str(p.resolve()) if p.exists() else str(p)
    if key not in seen and p.is_file():
        seen.add(key)
        videos.append(p)

if not videos:
    print("未找到 MP4。请确认 CLIP_VIDEO_ENABLED=true，且系统有 ffmpeg / imageio-ffmpeg。")
else:
    for vp in videos:
        print(f"MP4: {vp}  ({vp.stat().st_size} bytes)")
        # Jupyter 内嵌播放；若浏览器不支持可直接打开文件路径
        try:
            display(Video(filename=str(vp), embed=True, width=640))
        except Exception as exc:
            print("inline play failed:", exc)

## 4. ASR（音频 → 文本）

调用 `qwen3-asr-flash`（默认），结果写回 `clip.asr_text`，并进入后续 `speech_context_text()`。

In [ ]:
asr_meta = client.transcribe_clip(clip)
print(json.dumps(asr_meta, ensure_ascii=False, indent=2))

print("\nclip.asr_text =")
print(clip.asr_text or "(empty)")
print("\nspeech_context_text() 预览（ASR + events + mel_feature）:")
print(clip.speech_context_text()[:1500])

(OUT_DIR / f"{clip.clip_id}_asr.json").write_text(
    json.dumps(asr_meta, ensure_ascii=False, indent=2), encoding="utf-8"
)

## 5. Omni 打标（最终产物：标签）

`label_clip` 会使用：视频帧序列 + 音频 + ASR + events + **mel_feature_text** + taxonomy。

> 这里 `run_asr=False`，因为上一步已经跑过 ASR，避免重复计费。

In [ ]:
label_row = client.label_clip(clip, run_asr=False)

labels_path = OUT_DIR / f"{clip.clip_id}_labels.json"
labels_path.write_text(json.dumps(label_row, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved ->", labels_path)

print("\nscene_summary:")
print(label_row.get("scene_summary", ""))

print("\nlabels (keys):", list((label_row.get("labels") or {}).keys())[:20], "...")
print("mel_feature_included =", label_row.get("mel_feature_included"))
print("mel_matrix_path      =", label_row.get("mel_matrix_path"))

# 打印少量标签样例
sample_labels = dict(list((label_row.get("labels") or {}).items())[:5])
print("\nsample labels:")
print(json.dumps(sample_labels, ensure_ascii=False, indent=2))

## 6. Fusion Embedding（最终产物：向量 JSON）

输入侧通常包含：代表帧 + Mel PNG + ASR/events/**mel_feature_text** + `scene_summary`。

In [ ]:
embedding_row = client.embed_clip(
    clip,
    extra_text=label_row.get("scene_summary", ""),
)

# 向量本身很长：落盘完整 JSON，屏幕只打印摘要
emb_path = OUT_DIR / f"{clip.clip_id}_fusion_embedding.json"
emb_path.write_text(json.dumps(embedding_row, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved ->", emb_path)

vec = embedding_row.get("embedding") or []
inputs = embedding_row.get("inputs") or {}
print("model          =", embedding_row.get("model"))
print("dimension      =", embedding_row.get("dimension"), "actual_len=", len(vec))
print("embedding[:8]  =", vec[:8])
print("acoustic_panel =", inputs.get("acoustic_panel_path"))
print("mel_matrix     =", inputs.get("mel_matrix_path"))
print("mel_shape      =", inputs.get("mel_matrix_shape"))
text_preview = (inputs.get("text") or "")[:500]
print("text preview:\n", text_preview)

## 7. 一键全袋流水线（可选）

`process_bag` = extract → ASR → Omni 打标 → fusion embedding，并写出标准 jsonl：

- `labels.jsonl`
- `fusion_embeddings.jsonl`
- `clip_videos.jsonl`

适合批量；逐步调试请用上面 3–6 节。

In [ ]:
RUN_PROCESS_BAG = os.getenv("SDK_DEMO_RUN_PROCESS_BAG", "0") == "1"

if not RUN_PROCESS_BAG:
    print("跳过 process_bag（避免重复打 API）。")
    print("若要跑：设置环境变量 SDK_DEMO_RUN_PROCESS_BAG=1 后重跑本 cell。")
else:
    # 使用独立 work_dir，避免与逐步演示目录互相覆盖
    batch_work = OUT_DIR / "batch_work"
    batch_client = OmsMultimodalClient(
        taxonomy_path=bundled_taxonomy_path(),
        work_dir=batch_work,
        acoustic_panel_config=acoustic_cfg,
        clip_video_config=video_cfg,
        asr_config=asr_cfg,
        model_backend="api",
        storage_backend="local",
        load_dotenv=True,
    )
    output = OutputConfig(
        embeddings_out=OUT_DIR / "fusion_embeddings.jsonl",
        labels_out=OUT_DIR / "labels.jsonl",
        clips_out=OUT_DIR / "clips.jsonl",
        videos_out=OUT_DIR / "clip_videos.jsonl",
    )
    result = batch_client.process_bag(
        BAG_PATH,
        clip_config=clip_cfg,
        output=output,
    )
    print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2))
    print("\n请打开:")
    print(" -", output.labels_out)
    print(" -", output.embeddings_out)
    print(" -", output.videos_out)

## 8. 汇总：本 clip 产物清单

In [ ]:
def _ok(p: str | Path | None) -> str:
    if not p:
        return "MISSING"
    path = Path(p)
    return f"OK  {path}  ({path.stat().st_size} B)" if path.is_file() else f"MISSING  {path}"

summary = {
    "clip_id": clip.clip_id,
    "audio_wav": _ok(getattr(clip.audio, "audio_path", None) if clip.audio else None),
    "mel_png": _ok(clip.acoustic_panel_path),
    "mel_matrix_csv": _ok(clip.mel_matrix_path),
    "mel_matrix_meta": _ok(clip.mel_matrix_meta_path),
    "preview_mp4": _ok(clip.clip_video_path),
    "asr_text_chars": len(clip.asr_text or ""),
    "labels_json": _ok(OUT_DIR / f"{clip.clip_id}_labels.json"),
    "embedding_json": _ok(OUT_DIR / f"{clip.clip_id}_fusion_embedding.json"),
    "sdk_version": __version__,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

print("\noutput 目录:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:40} {p.stat().st_size:10} B")

---

### 常见问题

1. **`ModuleNotFoundError: oms_multimodal`**  
   在 `piplinesdk/` 执行 `pip install -e ".[mc]"` 或 `pip install -e .`，并确认 Jupyter kernel 是同一 venv。

2. **没有 `mel_matrix.csv`**  
   确认 SDK ≥ 0.3.2 且含 Mel 导出；`AcousticPanelConfig.export_mel_matrix=True`；clip 有音频。

3. **没有 MP4**  
   安装 ffmpeg，或 `pip install imageio-ffmpeg`；检查 `CLIP_VIDEO_ENABLED=true`。

4. **Omni / ASR 报错**  
   检查 `DASHSCOPE_API_KEY`、`DASHSCOPE_WORKSPACE_ID`（Omni 必需）。